---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！

# 06 - 完整示例：端到端数据加载

## 学习目标

1. 整合 Dataset、DataLoader、Sampler、Transform
2. 实现完整的数据加载流程
3. 处理真实场景中的常见问题
4. 掌握最佳实践

## 1. 完整数据加载流程

```
原始数据（硬盘）
    ↓
Dataset（定义如何读取单个样本）
    ↓
Transform（数据预处理和增强）
    ↓
Sampler（定义采样策略，可选）
    ↓
DataLoader（批量加载、多进程）
    ↓
模型训练
```

## 2. 示例场景：图像分类任务

### 任务描述

- 数据集：1000 张图像，3 个类别
- 类别不平衡：类别 0 (600张)、类别 1 (300张)、类别 2 (100张)
- 需要划分训练集和验证集
- 需要数据增强

## 3. 完整实现

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, random_split
from torchvision import transforms
from PIL import Image
import numpy as np
import collections

print("PyTorch version:", torch.__version__)

### 3.1 步骤 1：定义 Dataset

In [ ]:
class ImageClassificationDataset(Dataset):
    """图像分类数据集"""
    
    def __init__(self, num_samples_per_class, transform=None):
        """
        参数:
            num_samples_per_class: 列表，每个类别的样本数
            transform: 数据变换
        """
        self.transform = transform
        self.labels = []
        
        # 生成标签
        for class_idx, num_samples in enumerate(num_samples_per_class):
            self.labels.extend([class_idx] * num_samples)
        
        print(f"数据集创建完成：")
        print(f"  总样本数: {len(self.labels)}")
        print(f"  类别分布: {collections.Counter(self.labels)}")
    
    def __getitem__(self, idx):
        """获取单个样本"""
        # 创建随机图像（模拟真实场景）
        img_array = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
        img = Image.fromarray(img_array)
        
        # 应用 transform
        if self.transform is not None:
            img = self.transform(img)
        
        label = self.labels[idx]
        return img, label
    
    def __len__(self):
        return len(self.labels)
    
    def get_labels(self):
        """返回所有标签（用于创建 Sampler）"""
        return self.labels

# 创建不平衡数据集
num_samples_per_class = [600, 300, 100]  # 类别 0, 1, 2
full_dataset = ImageClassificationDataset(num_samples_per_class, transform=None)

print(f"\n类别分布统计:")
counter = collections.Counter(full_dataset.labels)
for cls, count in sorted(counter.items()):
    print(f"  类别 {cls}: {count} 个样本 ({count/len(full_dataset)*100:.1f}%)")

### 3.2 步骤 2：定义 Transform

In [ ]:
# 训练集 Transform（包含数据增强）
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 验证集 Transform（不做数据增强）
valid_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transform 定义完成：")
print("\n训练集 Transform:")
print(train_transform)
print("\n验证集 Transform:")
print(valid_transform)

### 3.3 步骤 3：划分训练集和验证集

In [ ]:
# 设置随机种子以保证可复现
torch.manual_seed(42)

# 划分数据集：80% 训练，20% 验证
train_size = int(0.8 * len(full_dataset))
valid_size = len(full_dataset) - train_size

train_dataset, valid_dataset = random_split(full_dataset, [train_size, valid_size])

print(f"数据集划分：")
print(f"  训练集: {len(train_dataset)} 个样本 ({len(train_dataset)/len(full_dataset)*100:.1f}%)")
print(f"  验证集: {len(valid_dataset)} 个样本 ({len(valid_dataset)/len(full_dataset)*100:.1f}%)")

# 为划分后的数据集设置 transform
# 注意：这里需要直接操作底层的 dataset
train_dataset.dataset.transform = train_transform
valid_dataset.dataset.transform = valid_transform

print("\nTransform 已分别设置")

### 3.4 步骤 4：创建 WeightedRandomSampler（处理不平衡）

In [ ]:
def get_weighted_sampler(dataset_subset, original_labels):
    """
    为不平衡数据集创建 WeightedRandomSampler
    
    参数:
        dataset_subset: random_split 返回的子集
        original_labels: 原始数据集的所有标签
    """
    # 获取子集的索引
    indices = dataset_subset.indices
    
    # 获取子集的标签
    subset_labels = [original_labels[i] for i in indices]
    
    # 统计类别分布
    label_counter = collections.Counter(subset_labels)
    print(f"子集类别分布: {label_counter}")
    
    # 计算类别权重（使用倒数）
    num_classes = len(label_counter)
    class_weights = torch.zeros(num_classes)
    for label, count in label_counter.items():
        class_weights[label] = 1.0 / count
    
    print(f"类别权重: {class_weights}")
    
    # 为每个样本分配权重
    sample_weights = [class_weights[label] for label in subset_labels]
    
    # 创建 sampler
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    
    return sampler

# 为训练集创建 WeightedRandomSampler
print("创建训练集 Sampler：")
train_sampler = get_weighted_sampler(train_dataset, full_dataset.labels)
print()

### 3.5 步骤 5：创建 DataLoader

In [ ]:
# 训练集 DataLoader（使用 sampler）
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    sampler=train_sampler,  # 使用加权采样
    num_workers=0,          # Windows 设为 0，Linux/Mac 可设为 4
    pin_memory=True,        # GPU 训练时加速
    drop_last=False
)

# 验证集 DataLoader（不使用 sampler，不打乱）
valid_loader = DataLoader(
    dataset=valid_dataset,
    batch_size=32,
    shuffle=False,          # 验证集不打乱
    num_workers=0,
    pin_memory=True,
    drop_last=False
)

print("DataLoader 创建完成：")
print(f"  训练集 batch 数量: {len(train_loader)}")
print(f"  验证集 batch 数量: {len(valid_loader)}")

## 4. 验证数据加载

In [ ]:
print("=== 测试训练集 DataLoader ===")
print()

# 获取一个 batch
for images, labels in train_loader:
    print(f"Batch 形状:")
    print(f"  images: {images.shape}")
    print(f"  labels: {labels.shape}")
    print(f"\nBatch 信息:")
    print(f"  图像数据类型: {images.dtype}")
    print(f"  图像数值范围: [{images.min():.3f}, {images.max():.3f}]")
    print(f"  标签内容: {labels.tolist()}")
    print(f"  标签分布: {collections.Counter(labels.tolist())}")
    break

print("\n=== 测试验证集 DataLoader ===")
print()

for images, labels in valid_loader:
    print(f"Batch 形状:")
    print(f"  images: {images.shape}")
    print(f"  labels: {labels.shape}")
    print(f"\nBatch 信息:")
    print(f"  标签内容: {labels.tolist()}")
    print(f"  标签分布: {collections.Counter(labels.tolist())}")
    break

## 5. 验证加权采样效果

In [ ]:
print("=== 验证加权采样效果 ===")
print()

# 统计多个 epoch 的采样分布
num_epochs = 3

for epoch in range(num_epochs):
    sampled_labels = []
    
    for images, labels in train_loader:
        sampled_labels.extend(labels.tolist())
    
    label_counter = collections.Counter(sampled_labels)
    print(f"Epoch {epoch + 1} 采样分布: {label_counter}")
    
    # 计算比例
    total = sum(label_counter.values())
    for cls in sorted(label_counter.keys()):
        ratio = label_counter[cls] / total * 100
        print(f"  类别 {cls}: {label_counter[cls]} ({ratio:.1f}%)")
    print()

print("通过加权采样，类别分布更加平衡！")

## 6. 模拟训练循环

In [ ]:
# 定义简单的模型
class SimpleModel(nn.Module):
    def __init__(self, num_classes=3):
        super(SimpleModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(16, num_classes)
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# 创建模型、损失函数、优化器
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleModel(num_classes=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"训练设备: {device}")
print(f"模型参数量: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# 训练函数
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # 前向传播
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 统计
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        # 打印前 3 个 batch
        if batch_idx < 3:
            print(f"  Batch {batch_idx}: loss={loss.item():.4f}")
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

# 验证函数
def validate(model, valid_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    avg_loss = total_loss / len(valid_loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

In [ ]:
# 训练循环
print("=== 开始训练 ===")
print()

num_epochs = 2

for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    
    # 训练
    print("  训练阶段:")
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    print(f"  训练 Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")
    
    # 验证
    print("  验证阶段:")
    valid_loss, valid_acc = validate(model, valid_loader, criterion, device)
    print(f"  验证 Loss: {valid_loss:.4f}, Accuracy: {valid_acc:.2f}%")
    print()

print("训练完成！")

## 7. 完整流程总结

### 7.1 数据准备流程

```python
# 1. 定义 Dataset
dataset = MyDataset(data_dir, transform=None)

# 2. 定义 Transform
train_transform = transforms.Compose([...])
valid_transform = transforms.Compose([...])

# 3. 划分数据集
train_set, valid_set = random_split(dataset, [train_size, valid_size])

# 4. 设置 Transform
train_set.dataset.transform = train_transform
valid_set.dataset.transform = valid_transform

# 5. 创建 Sampler（可选）
sampler = get_weighted_sampler(train_set, labels)

# 6. 创建 DataLoader
train_loader = DataLoader(train_set, batch_size=32, sampler=sampler)
valid_loader = DataLoader(valid_set, batch_size=32, shuffle=False)
```

### 7.2 最佳实践清单

#### Dataset
- ✅ 初始化时只读取元信息，不加载数据
- ✅ 在 `__getitem__` 中加载数据
- ✅ 实现 `__len__` 方法
- ✅ 支持 transform 参数

#### Transform
- ✅ 训练集使用数据增强
- ✅ 验证集只做基本预处理
- ✅ 使用 Compose 组合多个 transform
- ✅ 统一使用 ImageNet 的标准化参数

#### DataLoader
- ✅ 训练集 shuffle=True 或使用 sampler
- ✅ 验证集 shuffle=False
- ✅ 合理设置 batch_size（根据显存）
- ✅ 使用 num_workers 加速（Linux/Mac）
- ✅ GPU 训练时设置 pin_memory=True

#### Sampler
- ✅ 类别不平衡时使用 WeightedRandomSampler
- ✅ replacement=True（有放回采样）
- ✅ sampler 和 shuffle 不能同时使用

## 8. 常见问题排查

### 问题 1：数据加载太慢

In [ ]:
# 解决方案：
# 1. 增加 num_workers
train_loader = DataLoader(dataset, batch_size=32, num_workers=4)

# 2. 使用 pin_memory（GPU 训练）
train_loader = DataLoader(dataset, batch_size=32, pin_memory=True)

# 3. 减小图像分辨率
transform = transforms.Compose([
    transforms.Resize((128, 128)),  # 而不是 (224, 224)
    transforms.ToTensor()
])

print("数据加载优化建议已列出")

### 问题 2：显存不足 (OOM)

In [ ]:
# 解决方案：
# 1. 减小 batch_size
train_loader = DataLoader(dataset, batch_size=16)  # 从 32 降到 16

# 2. 减小图像分辨率
# 3. 减小模型大小
# 4. 使用梯度累积

print("OOM 问题解决方案已列出")

### 问题 3：训练不稳定

In [ ]:
# 解决方案：
# 1. 检查数据标准化
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])  # 必须标准化！
])

# 2. 使用加权采样处理不平衡数据
# 3. 设置随机种子
torch.manual_seed(42)

print("训练稳定性优化建议已列出")

## 9. 完整代码模板

以下是一个可直接使用的完整模板：

In [ ]:
# 完整的数据加载模板
def setup_data_loaders(data_dir, batch_size=32, num_workers=4):
    """
    设置训练和验证的 DataLoader
    
    参数:
        data_dir: 数据目录
        batch_size: 批量大小
        num_workers: 工作进程数
    
    返回:
        train_loader, valid_loader
    """
    # 1. 定义 Transform
    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    valid_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    # 2. 创建 Dataset
    # full_dataset = MyDataset(data_dir, transform=None)
    
    # 3. 划分数据集
    # train_size = int(0.8 * len(full_dataset))
    # valid_size = len(full_dataset) - train_size
    # train_dataset, valid_dataset = random_split(full_dataset, [train_size, valid_size])
    
    # 4. 设置 Transform
    # train_dataset.dataset.transform = train_transform
    # valid_dataset.dataset.transform = valid_transform
    
    # 5. 创建 DataLoader
    # train_loader = DataLoader(
    #     train_dataset,
    #     batch_size=batch_size,
    #     shuffle=True,
    #     num_workers=num_workers,
    #     pin_memory=True
    # )
    
    # valid_loader = DataLoader(
    #     valid_dataset,
    #     batch_size=batch_size,
    #     shuffle=False,
    #     num_workers=num_workers,
    #     pin_memory=True
    # )
    
    # return train_loader, valid_loader
    
    print("完整的数据加载模板")

print("代码模板已定义")

## 10. 总结

### 核心知识点

1. **数据加载流程**：Dataset → Transform → Sampler → DataLoader
2. **关键组件**：
   - Dataset：定义如何读取数据
   - Transform：数据预处理和增强
   - Sampler：采样策略（处理不平衡）
   - DataLoader：批量加载和并行

3. **最佳实践**：
   - 训练集使用数据增强
   - 验证集只做基本预处理
   - 类别不平衡时使用 WeightedRandomSampler
   - 合理设置 batch_size 和 num_workers

### 学习检查清单

- ✅ 能够自定义 Dataset
- ✅ 理解 Transform 的作用
- ✅ 会使用 DataLoader
- ✅ 能够处理不平衡数据
- ✅ 掌握完整的数据加载流程

### 下一步

恭喜！你已经完成了数据模块的学习。接下来可以：

1. 学习模型构建（第四章）
2. 学习训练技巧（第五章）
3. 在实际项目中应用这些知识

---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！